In [ ]:
import os

import pandas as pd
from style import *

os.chdir(ROOT)  # run from the project root, so data/ is found
apply_style()
banner(
    "What is in the data?",
    "Exploratory analysis of the goodbooks-10k ratings that Shelfmate learns from.",
)

In [ ]:
books = pd.read_csv("data/raw/books.csv").rename(columns={"authors": "author"})
ratings = pd.read_csv("data/raw/ratings.csv")
print(books.shape, ratings.shape)

In [ ]:
section(1, "Headline numbers", "How big is the data, and how empty is it?")
n_users, n_books = ratings["user_id"].nunique(), ratings["book_id"].nunique()
sparsity = 1 - len(ratings) / (n_users * n_books)
stat_tiles(
    {
        "Ratings": f"{len(ratings):,}",
        "Readers": f"{n_users:,}",
        "Books": f"{n_books:,}",
        "Average rating": f"{ratings['rating'].mean():.2f}",
        "Empty cells": f"{sparsity:.1%}",
    }
)

In [ ]:
section(
    2, "Is the data clean?", "Duplicates, missing values and ratings outside 1 to 5"
)
checks = pd.DataFrame(
    {
        "check": [
            "Duplicate reader and book pairs",
            "Ratings outside 1 to 5",
            "Books without an author",
            "Books without a publication year",
        ],
        "count": [
            ratings.duplicated(["user_id", "book_id"]).sum(),
            (~ratings["rating"].between(1, 5)).sum(),
            books["author"].isna().sum(),
            books["original_publication_year"].isna().sum(),
        ],
    }
)
show(checks, "Data quality")

In [ ]:
section(
    3,
    "How do people rate?",
    "Hover for exact values, drag to zoom, double-click to reset",
)
import plotly.express as px

counts = ratings["rating"].value_counts().sort_index()
share_45 = counts[[4, 5]].sum() / counts.sum()
fig = px.bar(x=counts.index, y=counts.values, text_auto=".2s")
fig.update_traces(
    hovertemplate="%{x} stars: %{y:,} ratings<extra></extra>",
    textposition="outside",
    cliponaxis=False,
)
interactive(
    fig,
    "Most ratings are 4 or 5 stars",
    f"{share_45:.0%} of ratings are 4 or 5. Number of ratings by star value.",
    "Stars",
    "Ratings",
    x={"dtick": 1},
    y={"tickformat": "~s"},
)

In [ ]:
section(
    4,
    "Who rates, and what gets rated?",
    "A few readers and a few books account for most ratings",
)
book_stats = (
    ratings.groupby("book_id")["rating"]
    .agg(n_ratings="size", avg_rating="mean")
    .join(books.set_index("book_id")[["title", "author", "original_publication_year"]])
)
user_stats = ratings.groupby("user_id")["rating"].agg(
    n_ratings="size", avg_rating="mean"
)

fig = px.histogram(user_stats, x="n_ratings", nbins=60, log_y=True)
fig.update_traces(hovertemplate="%{x} ratings: %{y:,} readers<extra></extra>")
interactive(
    fig,
    "Ratings per reader",
    f"Median {int(user_stats['n_ratings'].median()):,}. Log scale on the vertical axis.",
    "Ratings given by one reader",
    "Readers (log)",
)

ranked = book_stats.sort_values("n_ratings", ascending=False).reset_index(drop=True)
ranked["rank"] = ranked.index + 1
fig = px.line(ranked, x="rank", y="n_ratings", log_x=True, custom_data=["title"])
fig.update_traces(
    hovertemplate="<b>%{customdata[0]}</b><br>Rank %{x:,}<br>%{y:,} ratings<extra></extra>"
)
interactive(
    fig,
    "Ratings per book, most rated first",
    "The long tail: a few books hold most of the ratings",
    "Book rank (log scale)",
    "Ratings",
    y={"tickformat": "~s"},
)

In [ ]:
section(
    5, "Do popular books score higher?", "Average rating against the number of ratings"
)
import numpy as np

fig = px.density_heatmap(
    book_stats.assign(log_n=np.log10(book_stats["n_ratings"])),
    x="log_n",
    y="avg_rating",
    nbinsx=38,
    nbinsy=30,
    color_continuous_scale=SEQ_COLORS,
)
fig.update_traces(hovertemplate="Average rating %{y:.2f}<br>%{z} books<extra></extra>")
fig.update_coloraxes(colorbar={"title": "books", "thickness": 12, "outlinewidth": 0})
interactive(
    fig,
    "Popular books settle near 4 stars",
    "Each cell groups books by how many ratings they have and their average rating",
    "Ratings per book (log scale)",
    "Average rating",
    x={"tickvals": [2, 3, 4], "ticktext": ["100", "1K", "10K"]},
)

In [ ]:
section(
    6,
    "Which books do readers love most?",
    "Only books with at least 1,000 ratings, so a few fans cannot decide the list",
)
import plotly.graph_objects as go

top = book_stats[book_stats["n_ratings"] >= 1000].nlargest(15, "avg_rating").iloc[::-1]
left = top["avg_rating"].min() - 0.05
fig = go.Figure(
    go.Scatter(
        x=top["avg_rating"],
        y=top["title"].str.replace(r"\s*\(.*$", "", regex=True).str.slice(0, 40),
        mode="markers",
        marker={"size": 12, "color": BLUE, "line": {"width": 2, "color": SURFACE}},
        error_x={
            "type": "data",
            "symmetric": False,
            "array": np.zeros(len(top)),
            "arrayminus": top["avg_rating"] - left,
            "thickness": 1.2,
            "width": 0,
            "color": LINE,
        },
        customdata=top[["title", "n_ratings"]].values.tolist(),
        hovertemplate="<b>%{customdata[0]}</b><br>Average %{x:.2f}<br>%{customdata[1]:,} ratings<extra></extra>",
    )
)
interactive(
    fig,
    "The 15 highest rated books",
    "Average rating, books with at least 1,000 ratings",
    "Average rating",
    height=540,
    x={"range": [left, top["avg_rating"].max() + 0.03], "showgrid": True},
    y={"showgrid": False},
)

In [ ]:
section(
    7,
    "Which authors get the most attention?",
    "Total ratings across all of an author's books",
)
by_author = (
    ratings.merge(books[["book_id", "author"]], on="book_id")
    .groupby("author")["rating"]
    .agg(n_ratings="size", avg_rating="mean")
)
top_authors = by_author.nlargest(15, "n_ratings").iloc[::-1]

fig = px.bar(
    top_authors,
    x="n_ratings",
    y=top_authors.index.str.slice(0, 30),
    orientation="h",
    text_auto=".2s",
)
fig.update_traces(
    hovertemplate="%{y}: %{x:,} ratings<extra></extra>",
    textposition="outside",
    cliponaxis=False,
)
interactive(
    fig,
    "The 15 most rated authors",
    "Number of ratings",
    "Ratings",
    height=540,
    x={"tickformat": "~s", "showgrid": True},
    y={"showgrid": False},
)

In [ ]:
section(
    8,
    "Do older books rate differently?",
    "Average rating by decade of first publication",
)
decade = (
    book_stats.dropna(subset=["original_publication_year"])
    .assign(decade=lambda d: (d["original_publication_year"] // 10 * 10).astype(int))
    .groupby("decade")
    .agg(books=("title", "size"), avg_rating=("avg_rating", "mean"))
    .reset_index()
)
decade = decade[decade["books"] >= 20]

fig = px.line(decade, x="decade", y="avg_rating", markers=True, custom_data=["books"])
fig.update_traces(
    marker={"size": 9, "line": {"width": 2, "color": SURFACE}},
    hovertemplate="%{x}s: average %{y:.2f}<br>%{customdata[0]} books<extra></extra>",
)
interactive(
    fig,
    "Average rating by decade",
    "Decades with at least 20 books",
    "Decade of first publication",
    "Average rating",
    x={"dtick": 10},
)

In [ ]:
section(
    9,
    "Do series books rate higher?",
    "Books whose title ends in (Series name, #number)",
)
is_series = book_stats["title"].str.contains(r"\(.+?,\s*#[\d.]+\)\s*$", regex=True)
by_kind = (
    ratings.assign(
        kind=ratings["book_id"]
        .map(is_series)
        .map({True: "Series books", False: "Standalone books"})
    )
    .groupby("kind")["rating"]
    .agg(avg="mean", n="size")
)

fig = px.bar(
    by_kind.reset_index(),
    x="kind",
    y="avg",
    color="kind",
    color_discrete_sequence=[BLUE, ORANGE],
    text_auto=".2f",
    custom_data=["n"],
)
fig.update_traces(
    hovertemplate="%{x}: average %{y:.2f}<br>%{customdata[0]:,} ratings<extra></extra>",
    textposition="outside",
    cliponaxis=False,
)
fig.update_layout(showlegend=False, bargap=0.55)
interactive(
    fig,
    "Series and standalone books",
    "Average rating across all ratings",
    y={"range": [0, 5]},
)

In [ ]:
section(10, "Are some readers more generous?", "Each reader's own average rating")
generous = (user_stats["avg_rating"] >= 4.5).mean()
fig = px.histogram(user_stats, x="avg_rating", nbins=40)
fig.update_traces(hovertemplate="Average %{x:.2f}: %{y:,} readers<extra></extra>")
interactive(
    fig,
    "Readers differ in how they score",
    f"{generous:.1%} of readers average 4.5 or more",
    "Reader's average rating",
    "Number of readers",
    y={"tickformat": "~s"},
)

In [ ]:
section(
    11,
    "Where is the data thin?",
    "Books and readers with little history are hard to recommend for",
)
top_1pct = book_stats.nlargest(max(1, len(book_stats) // 100), "n_ratings")[
    "n_ratings"
].sum() / len(ratings)
stat_tiles(
    {
        "Books under 100 ratings": f"{(book_stats['n_ratings'] < 100).mean():.1%}",
        "Readers under 25 ratings": f"{(user_stats['n_ratings'] < 25).mean():.1%}",
        "Ratings on the top 1% of books": f"{top_1pct:.1%}",
    }
)

In [ ]:
hi = book_stats[book_stats["n_ratings"] >= 1000]["avg_rating"]
takeaways(
    [
        f"{share_45:.0%} of ratings are 4 or 5 stars, so a 3 already reads as a mild complaint.",
        f"The reader and book table is {sparsity:.1%} empty. A model has to predict cells nobody filled.",
        f"The median reader gave {int(user_stats['n_ratings'].median())} ratings. The median book got {int(book_stats['n_ratings'].median())}.",
        f"Books with 1,000 or more ratings average {hi.min():.2f} to {hi.max():.2f}. Popular books are rated well.",
        f"Series books average {by_kind.loc['Series books', 'avg']:.2f} and standalone books {by_kind.loc['Standalone books', 'avg']:.2f}.",
    ]
)